In [0]:
# Upgrade Databricks SDK to the latest version and restart Python to see updated packages
%pip install --upgrade databricks-sdk==0.70.0
%restart_python

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.jobs import JobSettings as Job

In [0]:
ROOT_PATH = '/Workspace/Users/icaro.data.eng@gmail.com/abinbev-challenge/' # Add your own path here

In [0]:
beverage_sales = Job.from_dict(
    {
        'name': 'beverage_sales',
        'tasks': [
            {
                'task_key': 'bronze-sales',
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}bronze/bronze_sales',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'bronze-channel_group',
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}bronze/bronze_channel_group',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-sales',
                'depends_on': [
                    {
                        'task_key': 'bronze-sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/silver_sales',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-channel_group',
                'depends_on': [
                    {
                        'task_key': 'bronze-channel_group',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/silver_channel_group',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'silver-sales_enriched',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                    {
                        'task_key': 'silver-channel_group',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}silver/sales_enriched',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_date',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_date',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_brand',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_brand',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_region',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_region',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_package',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_package',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-dim_channel',
                'depends_on': [
                    {
                        'task_key': 'silver-sales',
                    },
                    {
                        'task_key': 'silver-channel_group',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/dim_channel',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-fact_sales',
                'depends_on': [
                    {
                        'task_key': 'gold-dim_brand',
                    },
                    {
                        'task_key': 'gold-dim_region',
                    },
                    {
                        'task_key': 'gold-dim_channel',
                    },
                    {
                        'task_key': 'gold-dim_package',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/fact_sales',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-agg_sales_region_trade_group',
                'depends_on': [
                    {
                        'task_key': 'gold-fact_sales',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/agg_sales_region_trade_group',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-agg_sales_brand_month',
                'depends_on': [
                    {
                        'task_key': 'gold-fact_sales',
                    },
                    {
                        'task_key': 'gold-dim_date',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/agg_sales_brand_month',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'gold-agg_sales_region_brand_month',
                'depends_on': [
                    {
                        'task_key': 'gold-fact_sales',
                    },
                    {
                        'task_key': 'gold-dim_date',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}gold/agg_sales_region_brand_month',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'data_quality',
                'depends_on': [
                    {
                        'task_key': 'gold-fact_sales',
                    },
                    {
                        'task_key': 'gold-dim_date',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}data_quality',
                    'source': 'WORKSPACE',
                },
            },
            {
                'task_key': 'data_analysis',
                'depends_on': [
                    {
                        'task_key': 'data_quality',
                    },
                    {
                        'task_key': 'gold-agg_sales_region_trade_group',
                    },
                    {
                        'task_key': 'gold-agg_sales_brand_month',
                    },
                    {
                        'task_key': 'gold-agg_sales_region_brand_month',
                    },
                ],
                'notebook_task': {
                    'notebook_path': f'{ROOT_PATH}data_analysis',
                    'source': 'WORKSPACE',
                },
            },
        ],
        'queue': {
            'enabled': True,
        },
        'performance_target': 'PERFORMANCE_OPTIMIZED',
    }
)

In [0]:
w = WorkspaceClient()
# w.jobs.reset(new_settings=beverage_sales, job_id=<your job id>)
w.jobs.create(**beverage_sales.as_shallow_dict())